In [72]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import re

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

import seaborn as sns

### Importação de Dados

In [73]:
df = pd.read_csv('../data/raw/Personal_Finance_Dataset.csv')

In [74]:
df.head()

,Date,Transaction Description,Category,Amount,Type
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense
3,2020-01-05,Information last everything thank serve.,Investment,2291.00,Income
4,2020-01-13,Future choice whatever from.,Food & Drink,1126.88,Expense


In [75]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     1500 non-null   str    
 1   Transaction Description  1500 non-null   str    
 2   Category                 1500 non-null   str    
 3   Amount                   1500 non-null   float64
 4   Type                     1500 non-null   str    
dtypes: float64(1), str(4)
memory usage: 58.7 KB


### Renomeação e Mapeamento

In [76]:
df.columns

Index(['Date', 'Transaction Description', 'Category', 'Amount', 'Type'], dtype='str')

In [77]:
colunas_br = {
    'Date': 'Data',
    'Transaction Description': 'Descricao',
    'Category': 'Categoria',
    'Amount': 'Valor',
    'Type': 'Tipo transacao'
}

In [78]:
df = df.rename(columns=colunas_br)

In [79]:
categorias_br = {"Food & Drink": "Alimentação",
                 "Utilities": "Utilitários",
                 "Rent": "Aluguel",
                 "Investment": "Investimento",
                 "Shopping": "Compras",
                 "Health & Fitness": "Saúde",
                 "Entertainment": "Entretenimento",
                 "Travel": "Trajeto",
                 "Salary": "Salário",
                 "Other": "Outros"}

In [80]:
df['Categoria'] = df['Categoria'].map(categorias_br)

In [81]:
df

,Data,Descricao,Categoria,Valor,Tipo transacao
0,2020-01-02,Score each.,Alimentação,1485.69,Expense
1,2020-01-02,Quality throughout.,Utilitários,1475.58,Expense
2,2020-01-04,Instead ahead despite measure ago.,Aluguel,1185.08,Expense
3,2020-01-05,Information last everything thank serve.,Investimento,2291.00,Income
4,2020-01-13,Future choice whatever from.,Alimentação,1126.88,Expense
...,...,...,...,...,...
1495,2024-12-28,Quite as when.,Aluguel,514.09,Expense
1496,2024-12-28,Right analysis mention.,Entretenimento,727.25,Expense
1497,2024-12-28,No couple debate must.,Investimento,1425.00,Income
1498,2024-12-29,Discussion black follow.,Compras,655.78,Expense


In [82]:
tipo_transacao_br = {
    'Income': 'Receita',
    'Expense': 'Despesa'
}

In [83]:
df['Tipo transacao'] = df['Tipo transacao'].map(tipo_transacao_br)

In [84]:
df

,Data,Descricao,Categoria,Valor,Tipo transacao
0,2020-01-02,Score each.,Alimentação,1485.69,Despesa
1,2020-01-02,Quality throughout.,Utilitários,1475.58,Despesa
2,2020-01-04,Instead ahead despite measure ago.,Aluguel,1185.08,Despesa
3,2020-01-05,Information last everything thank serve.,Investimento,2291.00,Receita
4,2020-01-13,Future choice whatever from.,Alimentação,1126.88,Despesa
...,...,...,...,...,...
1495,2024-12-28,Quite as when.,Aluguel,514.09,Despesa
1496,2024-12-28,Right analysis mention.,Entretenimento,727.25,Despesa
1497,2024-12-28,No couple debate must.,Investimento,1425.00,Receita
1498,2024-12-29,Discussion black follow.,Compras,655.78,Despesa


In [85]:
df['Data'] = pd.to_datetime(df['Data'])

In [86]:
df['Data'] = df['Data'].dt.strftime('%d/%m/%Y')

In [87]:
df['Data']

0       02/01/2020
1       02/01/2020
2       04/01/2020
3       05/01/2020
4       13/01/2020
           ...    
1495    28/12/2024
1496    28/12/2024
1497    28/12/2024
1498    29/12/2024
1499    29/12/2024
Name: Data, Length: 1500, dtype: str

### Limpeza de Texto (REGEX)

In [88]:
def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = texto.strip()
    return texto

df['Descricao'] = df['Descricao'].apply(limpar_texto)

In [90]:
df['Descricao']

0                                    score each
1                            quality throughout
2             instead ahead despite measure ago
3       information last everything thank serve
4                   future choice whatever from
                         ...                   
1495                              quite as when
1496                     right analysis mention
1497                      no couple debate must
1498                    discussion black follow
1499           pressure activity defense detail
Name: Descricao, Length: 1500, dtype: str

### Vetorização (TF-IDF) e Classificação

In [91]:
X_train, X_test, y_train, y_test = train_test_split(df['Descricao'], df['Categoria'], test_size=0.2, random_state=2)

In [ ]:
pipeline